# selfplay_mcts_v3.ipynb -- Stage 4: DualNet (value + policy heads)

Stage 4 adds a policy head to the network. MCTS now uses policy head probabilities
as node priors instead of uniform priors, and training uses a dual loss.

| Dimension | Stage 3 (selfplay_mcts_v2) | Stage 4 (this notebook) |
|---|---|---|
| Network | ValueNet(768) -- value only | DualNet(768) -- value + policy |
| MCTS priors | Uniform over legal moves | Policy head probabilities |
| Loss | MSE(value, outcome) | MSE(value, outcome) + CE(policy, visit_counts) |
| Policy target | None | Normalized MCTS visit counts |

**Dual loss:** `loss = MSE(value_pred, outcome) + CE(policy_pred, visit_count_probs)`

CE is computed as `-sum(target_probs * log(pred_probs))` per position.
Policy target is the normalized MCTS visit count distribution at each position.

In [4]:
# Install / imports
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "chess", "torch"], check=False)

import os, sqlite3, json, time, random
from collections import deque
from datetime import datetime

import chess
import chess.pgn
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import MultiStepLR

import engine_nn_v2 as eng

PLANE_NAMES = eng.PLANE_NAMES
print(f"Plane names ({len(PLANE_NAMES)}): {PLANE_NAMES}")

Plane names (12): ['P', 'N', 'B', 'R', 'Q', 'K', 'p', 'n', 'b', 'r', 'q', 'k']


In [5]:
# Configuration
RUN_ID             = 32
N_GAMES            = 80
N_SIMULATIONS      = 200          # MCTS simulations per move
REPLAY_BUFFER_SIZE = 10_000
BATCH_SIZE         = 64
LR                 = 1e-3
WEIGHT_DECAY       = 1e-4
LR_MILESTONES      = [200, 500]   # absolute game counts where LR drops 10x
LR_GAMMA           = 0.1
TEMPERATURE        = 1.0          # sample proportional to visit counts during training
CHECKPOINT_EVERY   = 10
RESIGN_THRESHOLD   = 0.9        # resign when value < -this
RESIGN_CONSECUTIVE = 4          # consecutive half-moves below threshold
RESUME             = True

DB_PATH    = "../../data/chess_learning.db"
MODEL_DIR  = "../../models"
MODEL_PATH = f"{MODEL_DIR}/mcts_v3_run_{RUN_ID}.pt"

In [6]:
# Database setup
con = sqlite3.connect(DB_PATH)
cur = con.cursor()

# Add policy_loss column to game_records if not present
try:
    cur.execute("ALTER TABLE game_records ADD COLUMN policy_loss REAL")
    con.commit()
    print("Added policy_loss column to game_records")
except Exception:
    pass  # Column already exists

# Run metadata
NOTES          = f"mcts_v3_DualNet_768bitboard n_simulations={N_SIMULATIONS} batch={BATCH_SIZE} replay={REPLAY_BUFFER_SIZE}"
ENGINE_VERSION = "engine_nn_v2"

existing = cur.execute("SELECT run_id FROM runs WHERE run_id = ?", (RUN_ID,)).fetchone()
if existing and not RESUME:
    raise RuntimeError(f"run_id={RUN_ID} already exists. Set RESUME=True or choose a new RUN_ID.")
if not existing:
    cur.execute(
        "INSERT INTO runs (run_id, started_at, depth, learning_rate, move_limit, "
        "notes, last_updated, engine_version) VALUES (?, ?, ?, ?, ?, ?, ?, ?)",
        (RUN_ID, datetime.now().isoformat(), 0, LR, None,
         NOTES, datetime.now().isoformat(), ENGINE_VERSION)
    )
    con.commit()
    print(f"Created run_id={RUN_ID}")
else:
    print(f"Resuming run_id={RUN_ID}")

row = cur.execute("SELECT MAX(game_number) FROM games WHERE run_id = ?", (RUN_ID,)).fetchone()[0]
start_game = (row or 0) + 1
print(f"Starting at game_number={start_game}")

Resuming run_id=32
Starting at game_number=1


In [7]:
# Network, optimizer, replay buffer
os.makedirs(MODEL_DIR, exist_ok=True)

if RESUME and os.path.exists(MODEL_PATH):
    net = eng.load_model(MODEL_PATH, net_class=eng.DualNet, input_dim=768, hidden=256)
    print(f"Loaded model from {MODEL_PATH}")
else:
    net = eng.DualNet(input_dim=768, hidden=256)
    print("Initialized fresh DualNet(input_dim=768, hidden=256)")

optimizer = optim.AdamW(net.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

session_milestones = [max(1, m - start_game + 1) for m in LR_MILESTONES if m >= start_game]
scheduler = MultiStepLR(optimizer, milestones=session_milestones, gamma=LR_GAMMA)
print(f"LR milestones (session-relative): {session_milestones}")

# Replay buffer stores (board_tensor, value_target, legal_moves, policy_target)
# legal_moves: list[chess.Move] aligned with policy_target tensor
replay_buffer = deque(maxlen=REPLAY_BUFFER_SIZE)

Initialized fresh DualNet(input_dim=768, hidden=256)
LR milestones (session-relative): [200, 500]


In [8]:
# Self-play game using MCTS with DualNet policy priors
def play_game(net, n_simulations=N_SIMULATIONS, temperature=TEMPERATURE,
              resign_threshold=RESIGN_THRESHOLD, resign_consecutive=RESIGN_CONSECUTIVE):
    # Returns: examples [(board_tensor, value_target, legal_moves, policy_target)],
    #          board, n_halfmoves, terminated_by, resigned_turn
    board           = chess.Board()
    positions       = []  # (board_tensor, turn, legal_moves, policy_target)
    consecutive_bad = 0
    terminated_by   = "natural"
    resigned_turn   = None

    net.eval()
    while not board.is_game_over(claim_draw=True):
        tensor = eng.board_to_tensor(board)

        # Resignation check (DualNet returns value + policy)
        with torch.no_grad():
            _, val_out = net(tensor.unsqueeze(0))
            lv = val_out.squeeze().item()
        if lv < -resign_threshold:
            consecutive_bad += 1
            if consecutive_bad >= resign_consecutive:
                terminated_by = "resignation"
                resigned_turn = board.turn
                break
        else:
            consecutive_bad = 0

        legal_moves = list(board.legal_moves)
        positions.append((tensor, board.turn, legal_moves, None))  # policy filled below

        move, visit_counts = eng.mcts_search(
            board, net,
            n_simulations=n_simulations,
            temperature=temperature,
            # DualNet uses its own policy head as priors; board_to_input defaults to board_to_tensor
        )

        # Build policy target: normalized visit counts aligned with legal_moves
        total = sum(visit_counts.values()) or 1
        policy_target = torch.tensor(
            [visit_counts.get(m, 0) / total for m in legal_moves],
            dtype=torch.float32
        )
        positions[-1] = (tensor, board.turn, legal_moves, policy_target)
        board.push(move)

    if terminated_by == "resignation":
        white_val = -1.0 if resigned_turn == chess.WHITE else 1.0
    else:
        result    = board.result(claim_draw=True)
        white_val = {"1-0": 1.0, "0-1": -1.0}.get(result, 0.0)

    examples = [
        (tensor,
         white_val if turn == chess.WHITE else -white_val,
         legal_moves,
         policy_target)
        for tensor, turn, legal_moves, policy_target in positions
    ]
    return examples, board, len(positions), terminated_by, resigned_turn

In [9]:
# Dual-loss training step
# Iterates over batch items individually (legal_moves differ per position)
def train_step(net, optimizer, batch):
    net.train()
    optimizer.zero_grad()

    total_value_loss  = torch.tensor(0.0)
    total_policy_loss = torch.tensor(0.0)

    for board_tensor, value_target, legal_moves, policy_target in batch:
        policy_probs, value = net.forward(board_tensor, legal_moves)

        v_loss = F.mse_loss(value, torch.tensor(value_target, dtype=torch.float32))
        # Cross-entropy: -sum(target * log(pred)); clamp pred to avoid log(0)
        p_loss = -(policy_target * torch.log(policy_probs.clamp(min=1e-8))).sum()

        total_value_loss  = total_value_loss  + v_loss
        total_policy_loss = total_policy_loss + p_loss

    n    = len(batch)
    loss = (total_value_loss + total_policy_loss) / n
    loss.backward()
    nn.utils.clip_grad_norm_(net.parameters(), max_norm=1.0)
    optimizer.step()

    return loss.item(), (total_value_loss / n).item(), (total_policy_loss / n).item()

def eval_policy_loss(net, examples):
    # Compute mean policy loss for a game's positions (no gradient -- for logging only)
    if not examples:
        return 0.0
    net.eval()
    total = 0.0
    with torch.no_grad():
        for board_tensor, _, legal_moves, policy_target in examples:
            policy_probs, _ = net.forward(board_tensor, legal_moves)
            total += -(policy_target * torch.log(policy_probs.clamp(min=1e-8))).sum().item()
    return total / len(examples)

In [10]:
# Logging helpers
def board_to_pgn_str(board, game_number):
    game = chess.pgn.Game.from_board(board)
    game.headers["Event"] = "MCTS v3 Self-Play"
    game.headers["Round"] = str(game_number)
    return str(game)

def log_game(game_number, board, n_halfmoves, duration_s, policy_loss,
             terminated_by="natural", resigned_turn=None):
    if terminated_by == "resignation":
        outcome_str = "0-1" if resigned_turn == chess.WHITE else "1-0"
    else:
        result      = board.result(claim_draw=True)
        outcome_str = {"1-0": "1-0", "0-1": "0-1"}.get(result, "1/2-1/2")
    cur.execute(
        "INSERT INTO games (run_id, game_number, outcome, n_halfmoves, duration_s, terminated_by) "
        "VALUES (?, ?, ?, ?, ?, ?)",
        (RUN_ID, game_number, outcome_str, n_halfmoves, round(duration_s, 2), terminated_by)
    )
    cur.execute(
        "INSERT INTO game_records (run_id, game_number, pgn, policy_loss) VALUES (?, ?, ?, ?)",
        (RUN_ID, game_number, board_to_pgn_str(board, game_number), round(policy_loss, 6))
    )
    con.commit()
    return outcome_str

def log_weights(game_number, prev_weights):
    attr        = eng.attribution_weights(chess.Board(), net)
    scalar_attr = {k: v for k, v in attr.items() if isinstance(v, float)}
    for name, val in scalar_attr.items():
        cur.execute(
            "INSERT INTO weights (run_id, game_number, feature_name, weight_value) "
            "VALUES (?,?,?,?)",
            (RUN_ID, game_number, name, round(float(val), 6))
        )
        if prev_weights and name in prev_weights:
            cur.execute(
                "INSERT INTO weight_deltas (run_id, game_number, feature_name, delta) "
                "VALUES (?,?,?,?)",
                (RUN_ID, game_number, name, round(float(val) - prev_weights[name], 6))
            )
    con.commit()
    return scalar_attr

def log_feature_stats(game_number, outcome_str, examples):
    if not examples:
        return
    stacked     = torch.stack([x[0] for x in examples]).view(-1, 12, 64)
    plane_means = stacked.mean(dim=0).sum(dim=1).tolist()
    cur.executemany(
        "INSERT INTO feature_stats (run_id, game_number, outcome, feature_name, mean_value) "
        "VALUES (?,?,?,?,?)",
        [(RUN_ID, game_number, outcome_str, name, round(plane_means[i], 6))
         for i, name in enumerate(PLANE_NAMES)]
    )
    con.commit()

In [11]:
# Training loop
prev_weights   = None
decisive_count = 0

print(f"MCTS v3  RUN_ID={RUN_ID}  simulations={N_SIMULATIONS}  "
      f"games {start_game}-{start_game + N_GAMES - 1}")
print("-" * 80)

for game_idx in range(N_GAMES):
    game_number = start_game + game_idx
    t0 = time.time()

    examples, board, n_halfmoves, terminated_by, resigned_turn = play_game(net)
    replay_buffer.extend(examples)
    duration = time.time() - t0

    game_policy_loss = eval_policy_loss(net, examples)

    loss, v_loss, p_loss = None, None, None
    if len(replay_buffer) >= BATCH_SIZE:
        batch                = random.sample(replay_buffer, BATCH_SIZE)
        loss, v_loss, p_loss = train_step(net, optimizer, batch)
    scheduler.step()

    outcome_str = log_game(game_number, board, n_halfmoves, duration, game_policy_loss,
                           terminated_by=terminated_by, resigned_turn=resigned_turn)
    log_feature_stats(game_number, outcome_str, examples)
    if outcome_str != "1/2-1/2":
        decisive_count += 1

    if game_number % CHECKPOINT_EVERY == 0:
        prev_weights = log_weights(game_number, prev_weights)
        eng.save_model(net, MODEL_PATH)

    loss_str     = f"{loss:.4f}" if loss is not None else "  n/a "
    p_loss_str   = f"{p_loss:.4f}" if p_loss is not None else "  n/a "
    decisive_pct = decisive_count / (game_idx + 1) * 100
    print(f"G{game_number:4d} | {outcome_str:7s} | {n_halfmoves:3d} ply | "
          f"{duration:5.1f}s | loss={loss_str} p={p_loss_str} | "
          f"buf={len(replay_buffer):5d} | decisive={decisive_pct:4.1f}%")

print("-" * 80)
print(f"Done. Model saved to {MODEL_PATH}")
con.close()

MCTS v3  RUN_ID=32  simulations=200  games 1-80
--------------------------------------------------------------------------------
G   1 | 1/2-1/2 | 406 ply | 167.2s | loss=2.4760 p=2.4745 | buf=  406 | decisive= 0.0%
G   2 | 1/2-1/2 | 328 ply | 135.5s | loss=2.6197 p=2.6193 | buf=  734 | decisive= 0.0%
G   3 | 1/2-1/2 | 375 ply | 215.0s | loss=2.6611 p=2.6600 | buf= 1109 | decisive= 0.0%
G   4 | 1/2-1/2 | 416 ply | 270.7s | loss=2.9107 p=2.9101 | buf= 1525 | decisive= 0.0%
G   5 | 1/2-1/2 | 368 ply | 221.6s | loss=2.8097 p=2.8095 | buf= 1893 | decisive= 0.0%
G   6 | 1/2-1/2 | 317 ply | 153.7s | loss=3.0682 p=3.0680 | buf= 2210 | decisive= 0.0%
G   7 | 1/2-1/2 | 418 ply | 219.2s | loss=2.9802 p=2.9798 | buf= 2628 | decisive= 0.0%
G   8 | 1/2-1/2 | 258 ply | 128.0s | loss=2.7093 p=2.7090 | buf= 2886 | decisive= 0.0%
G   9 | 1/2-1/2 | 419 ply | 255.6s | loss=2.9965 p=2.9964 | buf= 3305 | decisive= 0.0%
G  10 | 1/2-1/2 | 328 ply | 186.6s | loss=2.8084 p=2.8082 | buf= 3633 | decisive= 0.0%
G